# Santorini V4 P2 training

Standalone Kaggle workflow for the fixed iteration-11 dose-escalation branch of the selected V4 6×192 canonical-D4 network. Attach the updated fixed V4 P2 runtime dataset and the iteration-11 run containing a resumable training checkpoint plus replay, select a P100 accelerator, then run all cells.

The runtime supplies fixed lineage inputs, the offline Linux oracle, and a digest-checked source fallback. The source cell can instead clone the repository. Training may stop early at a declared teacher-objective or oracle-ratchet review gate; the prior→target KL watch is telemetry only and never pauses training.

In [ ]:
# Routine controls
NUM_ITERATIONS = 6           # Fixed branch: iterations 12 through 17
RUN_NAME = "v4_p2_dose4_iter12_17" # New /kaggle/working directory name
REPLAY_REUSE = 4.0           # Only experimental knob changed from iteration 11
EXPECTED_RESUME_REPLAY_REUSE = 2.0
EXPECTED_START_ITERATION = 11
EXPECTED_END_ITERATION = 17
SNAPSHOT_INTERVAL = 1        # Flat checkpoint_N files every N iterations
ARENA_GAMES = 0              # Fresh milestone suites are run separately
RUN_END_ARENAS = False
PACKAGE_OUTPUTS = True       # Build one /kaggle/working/RUN_NAME.zip

# Source setup. `bundled` is offline and exactly matches the uploaded runtime.
# Use `git` after the desired training changes have been committed and pushed.
SOURCE_MODE = "bundled"    # `bundled` or `git`
REPOSITORY_URL = "https://github.com/Luminous9/alpha-zero-custom.git"
REPOSITORY_REF = "main"     # Branch, tag, or commit; resolved commit is recorded
INSTALL_MISSING_DEPENDENCIES = True
ENSURE_P100_COMPATIBLE_TORCH = True  # Replace CUDA 12.8 wheel with CUDA 12.6

# Usually leave blank. Auto-discovery picks the unique highest-iteration
# latest checkpoint under /kaggle/input and its sibling replay.
RESUME_CHECKPOINT = ""
RESUME_REPLAY = ""

# The validated production path is P100. Keep False for routine training.
ALLOW_NON_P100 = False

## 1. P100-compatible PyTorch bootstrap

PyTorch CUDA 12.8 wheels no longer contain Pascal `sm_60` kernels. Current Kaggle P100 sessions therefore need the same PyTorch release's official CUDA 12.6 wheel. Enable Kaggle Internet for this cell. It runs before importing `torch`, so no kernel restart is needed.

In [ ]:
import importlib.metadata
import subprocess
import sys

gpu_query = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    check=True, capture_output=True, text=True,
)
gpu_names = [line.strip() for line in gpu_query.stdout.splitlines() if line.strip()]
installed_torch = importlib.metadata.version("torch")
is_p100 = any("P100" in name.upper() for name in gpu_names)
if is_p100 and ENSURE_P100_COMPATIBLE_TORCH and "+cu126" not in installed_torch:
    if "torch" in sys.modules:
        raise RuntimeError("torch was imported before compatibility setup; restart the session and run all cells.")
    torch_release = installed_torch.split("+", 1)[0]
    compatible_torch = f"torch=={torch_release}+cu126"
    print(f"Replacing {installed_torch} with P100-compatible {compatible_torch} ...", flush=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet",
        "--upgrade", "--force-reinstall", compatible_torch,
        "--index-url", "https://download.pytorch.org/whl/cu126",
    ], check=True)
    installed_torch = importlib.metadata.version("torch")
print({"gpus": gpu_names, "torch_distribution": installed_torch})

## 2. GPU and Kaggle environment

In [ ]:
from pathlib import Path
import hashlib
import importlib
import importlib.util
import json
import os
import subprocess
import sys
import zipfile

import torch

INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
subprocess.run(["nvidia-smi"], check=False)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Enable a Kaggle GPU accelerator.")
gpu_name = torch.cuda.get_device_name(0)
if not ALLOW_NON_P100 and "P100" not in gpu_name.upper():
    raise RuntimeError(f"Select a P100 accelerator; Kaggle supplied {gpu_name!r}.")
cuda_arches = torch.cuda.get_arch_list()
if "P100" in gpu_name.upper() and "sm_60" not in cuda_arches:
    raise RuntimeError(f"Installed {torch.__version__} does not include P100 sm_60 kernels: {cuda_arches}")
cuda_smoke = (torch.ones(16, device="cuda") * 2).sum().item()
if cuda_smoke != 32.0:
    raise RuntimeError(f"CUDA smoke test returned {cuda_smoke}")
print(json.dumps({
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "cuda_arches": cuda_arches,
    "gpu": gpu_name,
}, indent=2))

## 3. Locate fixed runtime and prepare source

In [ ]:
manifests = list(INPUT_ROOT.rglob("v4-p2-runtime-manifest.json"))
if len(manifests) != 1:
    raise RuntimeError(f"Expected one V4 runtime manifest, found: {manifests}")
runtime_manifest = manifests[0]
runtime_root = runtime_manifest.parent
runner = runtime_root / "run_santorini_v4_p2_training_kaggle.py"
if not runner.is_file():
    raise FileNotFoundError(runner)

if SOURCE_MODE == "bundled":
    source_root = runtime_root
    source_commit = ""
elif SOURCE_MODE == "git":
    source_root = WORKING_ROOT / "alpha-zero-custom-source"
    if source_root.exists():
        if not (source_root / ".git").is_dir():
            raise RuntimeError(f"Existing source path is not a git checkout: {source_root}")
        subprocess.run(["git", "fetch", "--depth", "1", "origin", REPOSITORY_REF], cwd=source_root, check=True)
        subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=source_root, check=True)
    else:
        subprocess.run(["git", "clone", "--no-checkout", REPOSITORY_URL, str(source_root)], check=True)
        subprocess.run(["git", "fetch", "--depth", "1", "origin", REPOSITORY_REF], cwd=source_root, check=True)
        subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=source_root, check=True)
    source_commit = subprocess.run(
        ["git", "rev-parse", "HEAD"], cwd=source_root, check=True,
        capture_output=True, text=True,
    ).stdout.strip()
else:
    raise ValueError("SOURCE_MODE must be 'bundled' or 'git'.")

for required in ("main_santorini.py", "Coach.py", "MCTS.py", "arena_santorini_v4_p2_arm.py"):
    if not (source_root / required).is_file():
        raise FileNotFoundError(source_root / required)
print(json.dumps({
    "runtime_manifest": str(runtime_manifest),
    "source_mode": SOURCE_MODE,
    "source_root": str(source_root),
    "source_commit": source_commit or None,
}, indent=2))

## 4. Dependencies

P2 does not use the repository's legacy TensorFlow-era `requirements.txt`. This cell checks the small current runtime set and installs only missing non-Torch packages.

In [ ]:
dependency_imports = {
    "numpy": "numpy",
    "tqdm": "tqdm",
    "tensorboard": "tensorboard",
    "coloredlogs": "coloredlogs",
}
missing = [package for package, module in dependency_imports.items() if importlib.util.find_spec(module) is None]
if missing and INSTALL_MISSING_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
elif missing:
    raise RuntimeError(f"Missing dependencies: {missing}")
for module in dependency_imports.values():
    importlib.import_module(module)
subprocess.run(
    [sys.executable, "-c", "import numpy, torch, tqdm; from torch.utils.tensorboard import SummaryWriter; import main_santorini, Coach, MCTS; from santorini.pytorch.V4NNet import NNetWrapper"],
    cwd=source_root, check=True,
)
print("Dependencies and project imports are ready.")

## 5. Discover and validate resume inputs

In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def checkpoint_metadata(path):
    payload = torch.load(path, map_location="cpu", weights_only=False)
    metadata = payload.get("training_metadata", {})
    if metadata.get("training_mode") != "latest":
        raise ValueError(f"Not a resumable latest-mode checkpoint: {path}")
    return metadata

if RESUME_CHECKPOINT:
    resume_checkpoint = Path(RESUME_CHECKPOINT)
    resume_metadata = checkpoint_metadata(resume_checkpoint)
else:
    candidates = []
    for pattern in ("latest-training.pth.tar", "latest-training.pth.zip"):
        for path in INPUT_ROOT.rglob(pattern):
            try:
                metadata = checkpoint_metadata(path)
                candidates.append((int(metadata.get("iteration", -1)), path, metadata))
            except Exception as error:
                print(f"Ignoring non-resumable candidate {path}: {error}")
    if not candidates:
        raise RuntimeError("No resumable latest-training checkpoint found under /kaggle/input.")
    highest = max(item[0] for item in candidates)
    finalists = [item for item in candidates if item[0] == highest]
    distinct = {}
    for item in finalists:
        distinct.setdefault(sha256(item[1]), item)
    if len(distinct) != 1:
        raise RuntimeError(f"Multiple different iteration-{highest} checkpoints found; set RESUME_CHECKPOINT explicitly: {[str(item[1]) for item in finalists]}")
    _, resume_checkpoint, resume_metadata = next(iter(distinct.values()))

if RESUME_REPLAY:
    resume_replay = Path(RESUME_REPLAY)
else:
    replay_matches = [
        resume_checkpoint.parent / name
        for name in ("latest.examples.npz", "latest.examples.zip")
        if (resume_checkpoint.parent / name).is_file()
    ]
    if len(replay_matches) != 1:
        raise RuntimeError(f"Expected one sibling latest replay for {resume_checkpoint}, found: {replay_matches}")
    resume_replay = replay_matches[0]

if NUM_ITERATIONS < 1:
    raise ValueError("NUM_ITERATIONS must be positive.")
if REPLAY_REUSE <= 0:
    raise ValueError("REPLAY_REUSE must be positive.")
if int(resume_metadata["iteration"]) != EXPECTED_START_ITERATION:
    raise ValueError(f"This branch must start at iteration {EXPECTED_START_ITERATION}, found {resume_metadata['iteration']}.")
if int(resume_metadata.get("replay_reuse", -1)) != EXPECTED_RESUME_REPLAY_REUSE:
    raise ValueError(f"Expected a {EXPECTED_RESUME_REPLAY_REUSE}x ancestor, found {resume_metadata.get('replay_reuse')}.")
if int(resume_metadata["iteration"]) + NUM_ITERATIONS != EXPECTED_END_ITERATION:
    raise ValueError("This fixed branch must end at iteration 17.")
if SNAPSHOT_INTERVAL < 1:
    raise ValueError("SNAPSHOT_INTERVAL must be positive.")
if ARENA_GAMES < 0 or ARENA_GAMES % 2:
    raise ValueError("ARENA_GAMES must be nonnegative and even.")
if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME:
    raise ValueError("RUN_NAME must be one directory name.")
output_dir = WORKING_ROOT / RUN_NAME
if output_dir.exists() and any(output_dir.iterdir()):
    raise RuntimeError(f"Output directory is already nonempty: {output_dir}")

print(json.dumps({
    "resume_checkpoint": str(resume_checkpoint),
    "resume_checkpoint_sha256": sha256(resume_checkpoint),
    "resume_replay": str(resume_replay),
    "resume_iteration": int(resume_metadata["iteration"]),
    "requested_end_iteration": int(resume_metadata["iteration"]) + NUM_ITERATIONS,
    "output_dir": str(output_dir),
}, indent=2))

## 6. Train

In [ ]:
command = [
    sys.executable, str(runner),
    "--runtime-manifest", str(runtime_manifest),
    "--source-root", str(source_root),
    "--resume-checkpoint", str(resume_checkpoint),
    "--resume-replay", str(resume_replay),
    "--output-dir", str(output_dir),
    "--num-iterations", str(NUM_ITERATIONS),
    "--replay-reuse", str(REPLAY_REUSE),
    "--expected-resume-replay-reuse", str(EXPECTED_RESUME_REPLAY_REUSE),
    "--expected-start-iteration", str(EXPECTED_START_ITERATION),
    "--expected-end-iteration", str(EXPECTED_END_ITERATION),
    "--snapshot-interval", str(SNAPSHOT_INTERVAL),
    "--arena-games", str(ARENA_GAMES),
]
if source_commit:
    command += ["--expected-source-commit", source_commit]
if not RUN_END_ARENAS:
    command.append("--no-end-arenas")
if ALLOW_NON_P100:
    command.append("--allow-non-p100")
print("Launching:", " ".join(command), flush=True)
subprocess.run(command, check=True)

## 7. Review telemetry

`standard_prior_target_kl_mean` watches whether 96/32 search still changes the policy. The frozen deep-value fields measure paired drift from iteration 11 against the same 480 deep-oracle labels. One threshold breach warns; two consecutive breaches pause only after writing resumable outputs.

In [ ]:
contract_path = output_dir / "p2-training-contract.json"
contract = json.loads(contract_path.read_text())
print(json.dumps({
    key: contract[key]
    for key in (
        "status", "start_iteration", "requested_end_iteration",
        "completed_iterations", "last_iteration", "snapshots",
        "source", "arena_summary_by_anchor_iteration", "elapsed_seconds",
    )
}, indent=2))
rows = [
    json.loads(line)
    for line in (output_dir / "telemetry" / "telemetry.jsonl").read_text().splitlines()
    if line
]
columns = (
    "iteration", "value_target_mode", "value_target_beta",
    "v4_teacher_objective_step_delta",
    "v4_teacher_objective_cumulative_delta",
    "v4_seam_contrast_delta_from_baseline",
    "oracle_sparring_neural_win_rate", "oracle_sparring_rolling_score",
    "standard_prior_target_kl_count", "standard_prior_target_kl_mean",
    "standard_prior_target_kl_median",
    "standard_prior_target_kl_reference_ratio",
    "prior_target_kl_warning_streak", "prior_target_kl_warning",
    "v4_deep_value_overall_pearson_delta",
    "v4_deep_value_overall_mse_delta",
    "v4_deep_value_windows_9_11_pearson_delta",
    "v4_deep_value_windows_9_11_mse_delta",
    "v4_deep_value_warning_streak", "v4_deep_value_gate_triggered",
    "wall_total_seconds",
)
for row in rows:
    print({key: row.get(key) for key in columns})
print(f"Results ready at: {output_dir}")

## 8. Package one downloadable output

In [ ]:
if PACKAGE_OUTPUTS:
    archive_path = WORKING_ROOT / f"{RUN_NAME}.zip"
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
        for path in sorted(output_dir.rglob("*")):
            if path.is_file():
                archive.write(path, Path(RUN_NAME) / path.relative_to(output_dir))
    print(json.dumps({
        "download": str(archive_path),
        "bytes": archive_path.stat().st_size,
        "sha256": sha256(archive_path),
    }, indent=2))
    try:
        from IPython.display import FileLink, display
        display(FileLink(str(archive_path)))
    except ImportError:
        pass
else:
    print("PACKAGE_OUTPUTS is False; no archive created.")

## Continuing later

Download the single ZIP from `/kaggle/working`. This notebook is intentionally pinned to iteration 11 → 17 at 4× reuse; do not extend it or raise reuse in-place. We will review iteration 17 and prepare the next branch only if the gates and strength checks support it.